# Evaluate a golden codec on your own data

Score a published **compressionKIT** codec on signals **you** provide — either
your own recordings or the built-in synthetic generators (no dataset needed).

The flow is the same regardless of source:

1. Load a codec.
2. Get a 1-D signal sampled at `codec.sample_rate` Hz.
3. Split into `codec.frame_size` frames, round-trip, and aggregate metrics.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from compressionkit.evaluation.metrics import compute_signal_metrics
from compressionkit.runtime import load_codec
from compressionkit.synthetic import ecg_mcsharry, ppg_dynamical

CODEC_SOURCE = "Ambiq/compressionkit-ppg-4x"
# CODEC_SOURCE = "results/ppg_rvq_64hz_04x_golden/deploy"  # local, offline

codec = load_codec(CODEC_SOURCE)
fs, n = codec.sample_rate, codec.frame_size
print(f"{codec.name}  ({codec.modality}, {fs} Hz, {n}-sample frames)")

## Option A — synthetic signal (runs anywhere)

The analytical generators produce a clean, ground-truth waveform at the codec's
sample rate. Add calibrated noise to mimic real wearables.

In [ ]:
duration_s = 60.0
if codec.modality == "ecg":
    clean = ecg_mcsharry(duration_s=duration_s, sample_rate=float(fs), hr_mean=60.0)
else:
    clean = ppg_dynamical(duration_s=duration_s, sample_rate=float(fs), hr_mean=72.0)

# Optional: add noise at a target SNR to stress-test the codec.
# clean, _ = add_noise(clean, sample_rate=float(fs), snr_db=15.0)

signal = clean.astype("float32")
print(f"signal: {signal.shape[0]} samples ({signal.shape[0] / fs:.1f} s)")

## Option B — bring your own data

Provide a 1-D float32 array sampled at `codec.sample_rate` Hz (resample first if
your sensor runs at a different rate). Uncomment to use it instead of the
synthetic signal above.

In [ ]:
# signal = np.load("my_recording.npy").astype("float32")
# assert signal.ndim == 1, "expects a 1-D signal at codec.sample_rate Hz"
#
# If your data is at a different rate, resample to codec.sample_rate first:
# from scipy.signal import resample_poly
# signal = resample_poly(signal, fs, native_fs).astype("float32")

## Frame, round-trip, and score

In [ ]:
def to_frames(sig: np.ndarray, frame_size: int) -> np.ndarray:
    """Split a 1-D signal into non-overlapping (n_frames, frame_size) frames."""
    usable = (len(sig) // frame_size) * frame_size
    return sig[:usable].reshape(-1, frame_size)


frames = to_frames(signal, n)
rows, recon_frames = [], []
for i, f in enumerate(frames):
    enc = codec.compress(f)
    rec = codec.decompress(enc)
    recon_frames.append(rec)
    mm = compute_signal_metrics(f, rec)
    rows.append({
        "frame": i,
        "bits": enc.nbits,
        "prd_percent": mm["prd_percent"],
        "cosine_similarity": mm["cosine_similarity"],
        "rmse": mm["rmse"],
    })

df = pd.DataFrame(rows)
recon = np.concatenate(recon_frames)
overall_cr = (frames.size * 32) / df["bits"].sum()  # vs 32-bit float baseline

print(f"frames     : {len(df)}")
print(f"overall CR : {overall_cr:.2f}x (vs 32-bit float)")
df[["prd_percent", "cosine_similarity", "rmse"]].describe().loc[["mean", "50%", "min", "max"]]

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 5))

show = slice(0, min(5 * n, len(signal)))  # first ~5 frames
t = np.arange(len(recon))[show] / fs
ax[0].plot(t, signal[: len(recon)][show], label="original", lw=1.4)
ax[0].plot(t, recon[show], label="reconstruction", lw=1.1, alpha=0.85)
ax[0].set_title(f"{codec.name}  ·  median PRD {df['prd_percent'].median():.2f}%  ·  CR {overall_cr:.2f}x")
ax[0].set_xlabel("time (s)")
ax[0].legend()

ax[1].hist(df["prd_percent"], bins=30, color="#4C78A8")
ax[1].axvline(df["prd_percent"].median(), color="k", ls="--")
ax[1].set_xlabel("PRD %")
ax[1].set_ylabel("frames")
ax[1].set_title("Per-frame PRD distribution")

plt.tight_layout()
plt.show()

## Next steps

- Compare tiers (2×–32×) on the same signal to pick a CR/fidelity operating point.
- Stress-test robustness with `add_noise(..., snr_db=...)` at decreasing SNR.
- Set up real datasets via the [Dataset Setup](https://ambiqai.github.io/compressionkit/datasets/) guide for population-scale evaluation.